# Muzyka generowana przez Transformer: przewidywanie kolejnego zdarzenia

## Case study: Bach Chorales w symbolicznej reprezentacji MIDI

W tym notebooku budujemy mały decoder-only Transformer w PyTorch, który uczy się przewidywać kolejne zdarzenia muzyczne. Celem jest demonstracja kompletnego przepływu: reprezentacja muzyki, tokenizacja, okna treningowe, kodowanie pozycji, model z wieloma wejściami i wyjściami, trening teacher-forcing oraz autoregresywne próbkowanie.

### Dlaczego generowanie muzyki jest trudne?

Muzyka łączy lokalne zależności (kolejne nuty melodii, rytm) z długimi zależnościami (motyw, fraza, kadencja, powrót harmonii). MIDI przechowuje wykonanie jako zdarzenia czasowe, a nie jako prozę. Model musi zatem rozróżniać wysokość, czas, głos i czasem jednoczesne nuty. Wersja demonstracyjna upraszcza ten problem do kroków czasowych i klas wysokości.

### Autoregresja i faktoryzacja

Dla sekwencji tokenów $x_1,\ldots,x_T$ modeluje się rozkład jako $p(x_{1:T})=\prod_t p(x_t\mid x_{<t})$. W treningu wejście jest przesunięte o jeden krok względem celu. Maska przyczynowa blokuje uwagę na przyszłe pozycje. W generowaniu model dopisuje jeden token naraz, więc sampling jest sekwencyjny.

### Transformer, pozycja i wiele wyjść

Self-attention liczy podobieństwo zapytań i kluczy, a następnie miesza wartości. Multi-head attention pozwala równolegle uczyć się różnych relacji. Sam mechanizm uwagi nie zna kolejności; dodajemy sinusoidalne kodowanie pozycji. W tym przykładzie każdy krok jest wielowymiarowym wektorem bitów pianoroll, a wyjście przewiduje wszystkie 48 klas wysokości naraz. To wariant wielowyjściowy z niezależnymi logitami Bernoulliego, odpowiedni dla nut, które mogą brzmieć jednocześnie.

### Funkcja straty i ograniczenia

Użyjemy binary cross-entropy per krok i pitch. W przeciwieństwie do softmaxu, który wymusza dokładnie jedną klasę, sigmoid/BCE pozwala aktywować kilka nut równocześnie. Próg 0.5 daje prosty pianoroll; podczas generowania można dodać sampling Bernoulliego. Model nie jest profesjonalnym systemem kompozycji: krótki trening, mały korpus i uproszczona siatka rytmiczna służą wyłącznie do obserwacji efektu.

### Kodowanie polifonii

W reprezentacji typu pianoroll jedna kolumna czasowa może zawierać wiele nut, więc akordy nie wymagają sztucznego porządkowania nut jako pojedynczych tokenów. W realnym MIDI parser musi scalać zdarzenia note-on/note-off, rozwiązywać tempo i kwantyzować czas. W praktyce tokeny mogą kodować pitch, duration, velocity i instrument albo używać wielistreamowej reprezentacji.

**Parametry demonstracyjne:** mały model (d_model=96, 3 warstwy, 4 głowy), okna po 16 kroków i kilkaset aktualizacji. Na GPU i CPU powinno to działać w rozsądnym czasie; jakość dźwiękowa nie jest miarą jakości produkcyjnej.

## 1. Dane: JSB Chorales i reprezentacja pianoroll

Bach Chorales to klasyczny mały zbiór symboliczych partytur czterogłosowych. Każdy krok czasowy opisuje wysokości nut w głosach sopranu, altu, tenoru i basu. W tym case study pracujemy na syntetycznej, uproszczonej reprezentacji pianoroll: tensor `[czas, wysokość]`, gdzie 1 oznacza aktywną nutę. Loader najpierw próbuje pobrać plik CSV ze zbioru JSB Chorales. Jeśli sieć lub adres źródłowy jest niedostępny, generuje deterministyczne przykładowe sekwencje harmonii; komórka informuje, który wariant został użyty. To pozwala prześledzić cały pipeline bez ukrywania ograniczeń demonstracyjnego fallbacku.

Przy prawdziwym zbiorze zakładamy CSV z komórkami zawierającymi listy nut MIDI aktywnych w kolejnych krokach. Dla stabilności przykład mapuje nuty do 48 klas wysokości (MIDI 48–95), a sekwencje do stałej długości. Jest to uproszczenie: nie kodujemy długości nut, dynamiki, artykulacji ani dokładnego czasu.

**Co pokazuje ta komórka:** importuje zależności, ustawia ziarna losowości i buduje niewielki zbiór przykładowych partii w postaci wielogłosowych pianorolli. Możesz włączyć próbę pobrania JSB przez ustawienie `TRY_DOWNLOAD=True`.


In [ ]:
import math, random, urllib.request, csv, io
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED=23
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:",device)

PITCH_MIN, N_PITCH, STEPS = 48, 48, 32
TRY_DOWNLOAD=False  # Ustaw True, gdy chcesz spróbować pobrać publiczne dane.

def make_demo_corpus(n=240):
    # Cztery głosy, diatoniczne progresje i rytmiczne podtrzymania: kontrolowany fallback.
    scales=[0,2,4,5,7,9,11]
    roots=[0,5,7,0,9,5,0,7]
    rows=[]
    for _ in range(n):
        roll=np.zeros((STEPS,N_PITCH),dtype=np.float32)
        trans=random.choice([-2,0,0,0,2])
        for t in range(STEPS):
            beat=t//4; root=roots[beat%len(roots)]
            chord=[root,(root+4)%12,(root+7)%12,(root+11)%12]
            pitches=[60+root+trans, 60+chord[1]+trans, 48+chord[2]+trans, 48+root+trans]
            for voice,p in enumerate(pitches):
                p=max(PITCH_MIN,min(PITCH_MIN+N_PITCH-1,p))
                # Zmienność oktawy/tonu i okazjonalne pauzy imitują warianty partii.
                q=p + (12 if voice<2 and random.random()<.25 else 0)
                q=max(PITCH_MIN,min(PITCH_MIN+N_PITCH-1,q))
                if random.random()>.035: roll[t,q-PITCH_MIN]=1
        rows.append(roll)
    return np.stack(rows)

def load_jsb_or_demo():
    if TRY_DOWNLOAD:
        urls=["https://raw.githubusercontent.com/czhuang/JSB-Chorales-dataset/master/jsb-chorales-train.csv"]
        for url in urls:
            try:
                raw=urllib.request.urlopen(url,timeout=8).read().decode("utf-8")
                # Ten public format przechowuje po jednej obserwacji w wierszu.
                parsed=[]; sequence=[]
                for row in csv.reader(io.StringIO(raw)):
                    vals=[int(x) for x in row if x.strip().lstrip('-').isdigit()]
                    if not vals:
                        if len(sequence)>=STEPS:
                            for start in range(0,len(sequence)-STEPS+1,STEPS//2):
                                parsed.append(np.stack(sequence[start:start+STEPS]))
                        sequence=[]; continue
                    frame=np.zeros(N_PITCH,dtype=np.float32)
                    for p in vals:  # kilka głosów/nut w jednym kroku
                        if PITCH_MIN<=p<PITCH_MIN+N_PITCH: frame[p-PITCH_MIN]=1
                    sequence.append(frame)
                if len(sequence)>=STEPS:
                    for start in range(0,len(sequence)-STEPS+1,STEPS//2):
                        parsed.append(np.stack(sequence[start:start+STEPS]))
                if len(parsed)>20: return np.stack(parsed),"pobrane JSB Chorales (CSV; kroki wielogłosowe)"
            except Exception as e: print("Pobieranie nieudane:",type(e).__name__,str(e)[:120])
    return make_demo_corpus(),"fallback demonstracyjny (syntetyczne harmonie)"

rolls,data_source=load_jsb_or_demo()
print("Źródło:",data_source,"| liczba sekwencji:",len(rolls),"| kształt:",rolls.shape)
plt.figure(figsize=(10,2.5)); plt.imshow(rolls[0].T,aspect="auto",origin="lower",cmap="magma"); plt.xlabel("krok"); plt.ylabel("wysokość MIDI (indeks)" ); plt.title("Przykładowy pianoroll"); plt.show()


## 2. Przygotowanie sekwencji i podział danych

Sekwencje dzielimy na train i validation po całych przykładach, aby okna z tego samego utworu nie trafiały do obu części. Następnie uczymy na krótkich oknach wejściowych, a celem jest pianoroll przesunięty o jeden krok. Jest to odpowiednik teacher forcing: model dostaje prawdziwą historię podczas treningu.

**Co pokazuje ta komórka:** tworzy podział, konwersję do tensorów i losowy batch wejść oraz celów. Przy większych danych warto zachować oryginalny podział train/valid/test i dodać osobny test.


In [ ]:
perm=np.random.permutation(len(rolls)); cut=max(1,int(.85*len(rolls)))
train_np, val_np=rolls[perm[:cut]], rolls[perm[cut:]]
train=torch.tensor(train_np,device=device); val=torch.tensor(val_np,device=device)
CONTEXT=16; BATCH=24

def get_batch(split):
    data=train if split=="train" else val
    if len(data)==0: data=train
    ids=torch.randint(0,len(data),(BATCH,),device=device)
    starts=torch.randint(0,STEPS-CONTEXT-1,(BATCH,),device=device)
    x=torch.stack([data[i,s:s+CONTEXT] for i,s in zip(ids,starts)])
    y=torch.stack([data[i,s+1:s+CONTEXT+1] for i,s in zip(ids,starts)])
    return x,y
xb,yb=get_batch("train")
print("train/val:",len(train),len(val),"batch:",xb.shape,"target:",yb.shape)


## 3. Pozycja i causal self-attention

Sinusoidalne kodowanie pozycji dodaje do wektora cech informację o indeksie kroku. W multi-head attention każda głowa projektuje wejście do query, key i value. Maska dolnotrójkątna ustawia przyszłe pozycje na wartość bardzo ujemną przed softmaxem, dzięki czemu ich waga staje się zerowa.

**Co pokazuje ta komórka:** implementuje sinusoidy, uwagę wielogłową z maską przyczynową oraz blok pre-norm z residual connection. To pozwala zobaczyć elementy Transformer bez ukrywania ich za gotową biblioteką.


In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=256):
        super().__init__(); pos=torch.arange(max_len).float().unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2).float()*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_len,d_model); pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div[:pe[:,1::2].shape[1]])
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class CausalMHA(nn.Module):
    def __init__(self,d_model,heads,drop=.1):
        super().__init__(); assert d_model%heads==0
        self.h=heads; self.dh=d_model//heads
        self.qkv=nn.Linear(d_model,3*d_model); self.proj=nn.Linear(d_model,d_model); self.drop=nn.Dropout(drop)
    def forward(self,x):
        B,T,C=x.shape; q,k,v=self.qkv(x).chunk(3,dim=-1)
        def split(z): return z.view(B,T,self.h,self.dh).transpose(1,2)
        q,k,v=map(split,(q,k,v)); scores=(q@k.transpose(-2,-1))/math.sqrt(self.dh)
        mask=torch.triu(torch.ones(T,T,device=x.device,dtype=torch.bool),diagonal=1)
        scores=scores.masked_fill(mask,float("-inf")); a=self.drop(scores.softmax(-1))
        out=(a@v).transpose(1,2).contiguous().view(B,T,C)
        return self.proj(out)

class TransformerBlock(nn.Module):
    def __init__(self,d_model,heads,drop=.1):
        super().__init__(); self.n1=nn.LayerNorm(d_model); self.attn=CausalMHA(d_model,heads,drop)
        self.n2=nn.LayerNorm(d_model); self.ff=nn.Sequential(nn.Linear(d_model,4*d_model),nn.GELU(),nn.Dropout(drop),nn.Linear(4*d_model,d_model),nn.Dropout(drop))
    def forward(self,x):
        x=x+self.attn(self.n1(x)); return x+self.ff(self.n2(x))


## 4. Model: wielowymiarowe wejście i wyjście

Każdy krok pianorollu jest 48-wymiarowym wektorem wejściowym. Projekcja liniowa mapuje go do `d_model`. Stos Transformer przetwarza sekwencję, a głowica zwraca 48 logitów dla każdego kroku. Nie używamy embeddingu tokenów, bo wejściem nie jest pojedynczy identyfikator, tylko zestaw równoległych nut.

**Co pokazuje ta komórka:** definiuje model i sprawdza kształt wyjścia. Wynik `[batch, czas, wysokość]` jest zgodny z binarnym celem pianoroll.


In [ ]:
class MusicTransformer(nn.Module):
    def __init__(self,n_pitch=N_PITCH,d_model=96,heads=4,layers=3,drop=.1,max_len=128):
        super().__init__(); self.in_proj=nn.Linear(n_pitch,d_model); self.pos=SinusoidalPositionalEncoding(d_model,max_len)
        self.blocks=nn.ModuleList([TransformerBlock(d_model,heads,drop) for _ in range(layers)])
        self.norm=nn.LayerNorm(d_model); self.out=nn.Linear(d_model,n_pitch)
    def forward(self,x):
        z=self.pos(self.in_proj(x))
        for block in self.blocks: z=block(z)
        return self.out(self.norm(z))

model=MusicTransformer().to(device)
print("Parametry:",sum(p.numel() for p in model.parameters()))
print("logity:",model(xb).shape)


## 5. Trening i metryki

Optymalizujemy BCE-with-logits, która łączy sigmoid i binary cross-entropy stabilnie numerycznie. Raportujemy także aktywność (udział przewidywanych nut po progu 0.5), ponieważ model może osiągnąć deceptively niski loss, przewidując głównie ciszę. Krótki validation loss jest wskaźnikiem demonstracyjnym, a nie obiektywną oceną kompozycji.

**Co pokazuje ta komórka:** wykonuje ograniczony trening z clippingiem gradientu, pokazuje przebieg train/validation i liczbę aktualizacji. Zmniejsz `STEPS_TRAIN` dla wolnego CPU; zwiększaj dopiero po sprawdzeniu czasu.


In [ ]:
optimizer=torch.optim.AdamW(model.parameters(),lr=2e-3,weight_decay=1e-4)
STEPS_TRAIN=240; EVAL_EVERY=40
history={"step":[],"train":[],"val":[]}
@torch.no_grad()
def eval_loss(n_batches=8):
    model.eval(); vals=[]
    for _ in range(n_batches):
        x,y=get_batch("val"); vals.append(F.binary_cross_entropy_with_logits(model(x),y).item())
    model.train(); return float(np.mean(vals))
for step in range(1,STEPS_TRAIN+1):
    x,y=get_batch("train"); logits=model(x); loss=F.binary_cross_entropy_with_logits(logits,y)
    optimizer.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
    if step%EVAL_EVERY==0 or step==1:
        vl=eval_loss(); history["step"].append(step); history["train"].append(loss.item()); history["val"].append(vl)
        print(f"krok {step:3d} | train BCE {loss.item():.4f} | val BCE {vl:.4f}")
plt.plot(history["step"],history["train"],marker="o",label="train"); plt.plot(history["step"],history["val"],marker="o",label="validation")
plt.xlabel("aktualizacja"); plt.ylabel("BCE"); plt.grid(); plt.legend(); plt.show()


## 6. Generowanie i analiza pianorollu

Podczas generacji przekazujemy początkowy fragment, bierzemy prognozę dla następnego kroku i dopisujemy ją do kontekstu. Tutaj używamy próbkowania Bernoulliego z prawdopodobieństw sigmoid. `temperature` skaluje logity: niższa daje bardziej zdecydowane prognozy, wyższa zwiększa niepewność. Pokazujemy także aktywność i prosty wykres pianoroll.

**Co pokazuje ta komórka:** autoregresyjnie tworzy kilkanaście kroków i zestawia kontekst rzeczywisty z wygenerowanym fragmentem. Ponieważ trening jest krótki, wynik może brzmieć chaotycznie; kluczowe jest prześledzenie mechaniki.


In [ ]:
@torch.no_grad()
def generate(seed_roll,steps=16,temperature=1.0,threshold=None):
    model.eval(); context=torch.tensor(seed_roll[:CONTEXT],device=device).unsqueeze(0)
    generated=[]
    for _ in range(steps):
        logits=model(context[:,-CONTEXT:])[:,-1,:]/temperature
        probs=logits.sigmoid()
        nxt=torch.bernoulli(probs) if threshold is None else (probs>threshold).float()
        generated.append(nxt[0].cpu().numpy()); context=torch.cat([context,nxt[:,None,:]],dim=1)
    model.train(); return np.stack(generated)
seed=val_np[0] if len(val_np) else train_np[0]
gen=generate(seed,steps=16,temperature=.85)
fig,ax=plt.subplots(1,2,figsize=(11,3))
ax[0].imshow(seed[:16].T,aspect="auto",origin="lower",cmap="magma"); ax[0].set_title("Kontekst z danych")
ax[1].imshow(gen.T,aspect="auto",origin="lower",cmap="magma"); ax[1].set_title("Kontynuacja modelu")
for a in ax: a.set_xlabel("krok"); a.set_ylabel("indeks wysokości")
plt.tight_layout(); plt.show()
print("Średnia liczba aktywnych nut/krok:",round(float(gen.sum(1).mean()),2))


## Key takeaways

- Transformer jest autoregresyjnym modelem warunkowym: przewiduje kolejny krok z dotychczasowej historii.
- Maska causal attention blokuje przeciek informacji z przyszłości; positional encoding dostarcza informacji o kolejności.
- Pianoroll koduje polifonię przez wiele aktywnych wysokości w tym samym kroku. BCE-with-logits pasuje do wielu niezależnych etykiet.
- Trening teacher-forcing jest równoległy po pozycjach, natomiast generowanie pozostaje sekwencyjne.
- Krótki trening na uproszczonym lub syntetycznym korpusie pokazuje mechanikę, ale nie jakość kompozycji. Przy porównaniach należy jawnie podać źródło danych, podział, reprezentację i sposób próbkowania.
- Następne rozszerzenia: parsing MIDI (note-on/off, tempo, instrument), tokeny czasu trwania i velocity, dłuższy kontekst, kontrola tonacji oraz ocena z użyciem osobnych utworów testowych.